# Assignment 27 : ChatBots with Convarsation History using Langchain

# PART 1 -- Messages & Message Placeholders

# Task 1 : Understanding Chat Messages
## Answer Briefly:
## 1. What are SystemMessage, HumanMessage, and AIMessage ?
## 2. Why message-based prompting is better than single prompts?

# 1. What are SystemMessage, HumanMessage, and AIMessage?
## SystemMessage: Defines the AI's role, behavior, and instructions.
## HumanMessage: Represents the message or question sent by the user.
## AIMessage: Represents the response generated by the AI.

# Task 2: MessagePlaceholder Usage
## 1. Create a ChatPromptTemplate.
## 2. Use MessagesPlaceholder for chat history.
## 3. Inject dynamic conversation history into the prompt.
## 4. Test with 2-3 message turns.

In [1]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# 1. Create chat prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# 2. Create conversation history
chat_history = [
    HumanMessage(content="My name is Ani."),
    AIMessage(content="Hello Ani! Nice to meet you."),
    HumanMessage(content="I am learning Python."),
    AIMessage(content="That's great! Python is useful for AI.")
]

# 3. Inject dynamic history into prompt
messages = prompt.format_messages(
    chat_history=chat_history,
    question="Which programming language am I learning?"
)

# 4. Print messages
for message in messages:
    print(f"{message.type}: {message.content}")


system: You are a helpful assistant.
human: My name is Ani.
ai: Hello Ani! Nice to meet you.
human: I am learning Python.
ai: That's great! Python is useful for AI.
human: Which programming language am I learning?


# PART 2 -- Conversation History Management

# Task 3 : Basic Message History
## 1. Store conversation messages in a list.
## 2. Append user and AI messages after interaction.
## 3. Pass full history to the LLM.

In [2]:

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# 1. Initialize LLM
llm = ChatOllama(model="llama3.2")

# 2. Store conversation history
chat_history = [
    SystemMessage(content="You are a helpful assistant.")
]

# 3. Chat loop
while True:
    question = input("You: ")

    if question.lower() == "exit":
        break

    # Append user message
    chat_history.append(HumanMessage(content=question))

    # Pass full history to LLM
    response = llm.invoke(chat_history)

    # Append AI message
    chat_history.append(AIMessage(content=response.content))

    print("AI:", response.content)


AI: I'm an artificial intelligence model, which means I'm a computer program designed to simulate human-like conversations and answer questions to the best of my knowledge. I was trained on a massive dataset of text from various sources, including books, articles, and online content.

I don't have a personal history, emotions, or physical presence like a human would. I exist solely as a digital entity, designed to provide information and assist with tasks to the best of my abilities.

I'm constantly learning and improving my responses based on the interactions I have with users like you. I can provide information on a wide range of topics, from science and history to entertainment and culture.

One of my strengths is my ability to understand and respond to natural language, which means I can engage in conversations that feel natural and human-like. However, I'm not perfect and can make mistakes or not fully understand the context of a question or conversation.

I'm here to help and pro

# Task 4: Trimming Chat History
## 1. Implement a function to:
## . Limit number of past messages
## OR limit total token length
## 2. Trim older messages when limit is exceeded.
## 3. Verify chatbot still responds correctly.

In [3]:

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

# Initialize LLM
llm = ChatOllama(model="llama3.2")

# Maximum number of history messages
MAX_MESSAGES = 6

# Store conversation
chat_history = [
    SystemMessage(content="You are a helpful assistant.")
]

# Function to trim old messages
def trim_history(history):
    system_message = history[0]
    messages = history[1:]

    # Keep only the latest 6 messages
    messages = messages[-MAX_MESSAGES:]

    return [system_message] + messages

# Chat loop
while True:
    question = input("You: ")

    if question.lower() == "exit":
        break

    # Store user message
    chat_history.append(HumanMessage(content=question))

    # Trim history before sending to LLM
    chat_history = trim_history(chat_history)

    # Generate response
    response = llm.invoke(chat_history)

    # Store AI response
    chat_history.append(AIMessage(content=response.content))

    print("AI:", response.content)
    print("History messages:", len(chat_history) - 1)


# PART 3-- Q&A Chatbot with Message History

# Task 5: Build Q&A Chatbot
## Create a Chatbot that:
## 1. Accepts user questions
## 2. Uses message hsitory for context
## 3. Responds accurately based on previous conversation Test wiht follow-up questions
## " Expalain Python lists"
## "Give an Example"
## "What about tuples?"

In [4]:

from langchain_ollama import ChatOllama
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

# Initialize LLM
llm = ChatOllama(model="llama3.2", temperature=0)

# Store conversation history
chat_history = [
    SystemMessage(
        content="You are a helpful assistant. "
                "Use conversation history to understand "
                "follow-up questions and answer accurately."
    )
]

# Chatbot loop
while True:
    question = input("\nYou: ")

    if question.lower() == "exit":
        break

    # Add user question
    chat_history.append(HumanMessage(content=question))

    # Generate response using full history
    response = llm.invoke(chat_history)

    # Save AI response
    chat_history.append(
        AIMessage(content=response.content)
    )


    print("AI:", response.content)

print("Chatbot closed.")


AI: **Python Lists**

Python lists are ordered collections of items that can be of any data type, including strings, integers, floats, and other lists. They are denoted by square brackets `[]` and are mutable, meaning they can be modified after creation.

**Creating a List**
-----------------

You can create a list in several ways:

*   Using square brackets `[]` with elements separated by commas: `my_list = [1, 2, 3, 4, 5]`
*   Using the `list()` function: `my_list = list((1, 2, 3, 4, 5))`
*   Using a list comprehension: `my_list = [x for x in range(1, 6)]`

**List Operations**
-----------------

Here are some common operations you can perform on lists:

*   **Indexing**: Access a specific element in the list using its index. Indexing starts at 0. `my_list[0]` would return the first element.
*   **Slicing**: Extract a subset of elements from the list using square brackets `[]` with a start and end index. `my_list[1:3]` would return the second and third elements.
*   **Append**: Add an

# Task 6: Build Stateful Chatbot Application
## Build a chatbot that:
## Maintains conversation history
## Trims old messages
## Answers Questions contextually

In [ ]:

from langchain_ollama import ChatOllama
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

# 1. Initialize LLM
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# 2. Set history limit
MAX_MESSAGES = 6

# 3. Initialize conversation history
chat_history = [
    SystemMessage(
        content="You are a helpful assistant. "
                "Use conversation history to understand "
                "follow-up questions."
    )
]

# 4. Function to trim old messages
def trim_history(history):
    system_message = history[0]
    messages = history[1:]

    # Keep latest 6 messages
    messages = messages[-MAX_MESSAGES:]

    return [system_message] + messages

# 5. Chatbot application

while True:
    question = input("\nYou: ").strip()

    if question.lower() == "exit":
        break

    if not question:
        continue

    # Add user message
    chat_history.append(
        HumanMessage(content=question)
    )

    # Trim older messages
    chat_history = trim_history(chat_history)

    # Generate contextual response
    response = llm.invoke(chat_history)

    # Store AI response
    chat_history.append(
        AIMessage(content=response.content)
    )

    print("\nAI:", response.content)

    print(
        "History messages:",
        len(chat_history) - 1
    )



Chatbot is ready! Type 'exit' to stop.

AI: Python is a high-level, interpreted programming language that is widely used for various purposes such as web development, scientific computing, data analysis, artificial intelligence, and more.

Here are some key features that make Python a popular choice:

1. **Easy to learn**: Python has a simple syntax and is relatively easy to read and write, making it a great language for beginners.
2. **High-level language**: Python is a high-level language, meaning it abstracts away many low-level details, allowing you to focus on the logic of your program without worrying about memory management, etc.
3. **Interpreted language**: Python code is interpreted, meaning that it is executed line by line, rather than compiled all at once.
4. **Large standard library**: Python has a vast collection of libraries and modules that make it easy to perform various tasks, such as file I/O, networking, and data analysis.
5. **Cross-platform**: Python can run on mul

# Task 7: Observations & Insights
## write short answers:
## 1. Why Chat history is Important
## 2. Trade-offs between long memory and performance

# 1. Why is Chat History Important?

## Chat history helps the chatbot remember previous questions and answers, understand follow-up questions, and provide more relevant, contextual responses.

# 2. Trade-offs Between Long Memory and Performance

## Longer chat history provides more context but increases token usage, processing time, and computational cost. Shorter history improves performance but may lose important information.